# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MEDOXIII/FlyRank-Internship-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I chose Logistic Regression as a simple, auditable first learned benchmark for this ranking task. The target is a binary proxy label, but the operational goal is to rank pages for human review rather than make a hard yes/no decision for every page. The model score is used to order the queue; it has not been calibrated and should not be read as a reliable probability.

The scoring features exclude the direct label sources (`trend_direction` and `trend_pct`), the target itself, and the content/client IDs. IDs are used only for grouping or deterministic tie-breaking. `avg_position = 0` is treated as missing, imputed with the training-set median, and accompanied by a missingness indicator. This avoids direct label leakage, but it does not remove the time-window overlap: the label is based on the change from the preceding 30 days to the latest 30 days, while several features summarize a 90-day window. Results therefore describe within-window associations, not future performance or causal effects.

In [83]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

RANDOM_SEED = 42

repo_root = None
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "scripts" / "ml_utils.py").exists():
        repo_root = candidate
        break
if repo_root is None:
    repo_root = Path.cwd()

scripts_dir = repo_root / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Data contract and leakage guardrails
DATA_PATH = repo_root / "data" / "processed" / "refresh_feature_vector.csv"
df = pd.read_csv(DATA_PATH)

assert "is_declining_label" in df.columns, "Target column is missing"
assert "trend_direction" in df.columns, "Label source field is missing"
assert "content_id" in df.columns and "client_id" in df.columns, "IDs are missing"

# Zero means no position data, not a valid rank.
position_missing_mask = pd.to_numeric(df["avg_position"], errors="coerce").eq(0)
avg_position_missing_count = int(position_missing_mask.sum())
assert avg_position_missing_count > 0, "Expected avg_position=0 missing-value codes were not found"
df.loc[position_missing_mask, "avg_position"] = np.nan

# Exclude direct label sources, IDs, and overlapping 30-day windows from model inputs.
exclude = ["content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"]
feature_columns = [c for c in df.columns if c not in exclude]

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

missing = [c for c in numeric_features + categorical_features if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected feature columns: {missing}")

forbidden_features = set(exclude) | {
    "measurable_opportunity",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
}
model_features = set(numeric_features + categorical_features)
assert model_features.issubset(set(feature_columns)), "A selected model feature is absent from the feature vector"
assert model_features.isdisjoint(forbidden_features), (
    f"Forbidden columns are included as model features: {model_features & forbidden_features}"
)
print("Model feature allowlist and leakage assertion: passed")

# Hold out clients, not rows, to test cross-client generalization.
client_series = df["client_id"].astype(str)
client_values = np.array(client_series.drop_duplicates())
rng = np.random.default_rng(RANDOM_SEED)
shuffled_clients = rng.permutation(client_values)
test_client_count = max(1, round(len(shuffled_clients) * 0.2))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = client_series.isin(test_clients)
train_index = np.where(~test_mask)[0]
test_index = np.where(test_mask)[0]

X_train = df.iloc[train_index][numeric_features + categorical_features].copy()
X_test = df.iloc[test_index][numeric_features + categorical_features].copy()
y_train = df.iloc[train_index]["is_declining_label"].astype(int)
y_test = df.iloc[test_index]["is_declining_label"].astype(int)

print(f"Random seed: {RANDOM_SEED}")
print(f"Rows: {len(df):,} | full-data base rate: {df['is_declining_label'].mean():.3f}")
print(f"Clients: {len(client_values):,} total; {len(test_clients):,} held out for test")
print(
    f"Holdout rows: {len(test_index):,} ({len(test_index) / len(df):.1%} of pages) | "
    f"positive rate in test: {y_test.mean():.3f}"
)
print(f"Train rows: {len(train_index):,} | test rows: {len(test_index):,}")
print(f"avg_position zeros retained as missing: {avg_position_missing_count:,}")

Model feature allowlist and leakage assertion: passed
Random seed: 42
Rows: 30,000 | full-data base rate: 0.542
Clients: 32 total; 6 held out for test
Holdout rows: 2,325 (7.8% of pages) | positive rate in test: 0.391
Train rows: 27,675 | test rows: 2,325
avg_position zeros retained as missing: 1,205


## 2. Split design

I use a client-grouped holdout selected with random seed 42. Six of the 32 clients, containing 2,325 of the 30,000 pages (7.8%), are held out; no pages from those clients appear in training. The small page share reflects the uneven number of pages per client, so the split is grouped by client rather than by row.

The Week-4 rule is re-evaluated on these same held-out pages, using its deterministic tie-breakers, so the ranking comparison uses identical test cases. A time-aware split is not possible because this starter dataset has no calendar dates. Since this is one random client holdout, the observed differences are specific to this split and should not be treated as stable performance differences across all client samples.

In [84]:
# Check the grouped split size and the holdout label rate.
print(f"Client holdout test set size: {len(test_index):,} rows from {len(test_clients):,} clients")
print(f"Holdout share of all pages: {len(test_index) / len(df):.1%}")
print(f"Positive rate in holdout: {y_test.mean():.3f}")
print(f"Full-data base rate: {df['is_declining_label'].mean():.3f}")

Client holdout test set size: 2,325 rows from 6 clients
Holdout share of all pages: 7.8%
Positive rate in holdout: 0.391
Full-data base rate: 0.542


## 3. Train + compare vs my baseline

The model and Week-4 rule are evaluated on the same client-held-out pages using Precision@K. The rule uses Week-4's deterministic tie-breakers: impressions, age, then `content_id`; the model uses `content_id` only to break ties in its score. `avg_position` zeros are represented as missing and imputed using the training-set median (11.8), with a missingness indicator.

On this holdout, Precision@20 is 0.400 for Logistic Regression (8 of 20 pages) and 0.350 for the rule (7 of 20). At Precision@50, the model reaches 0.460 (23 pages) versus 0.300 (15 pages) for the rule; at Precision@100, the values are 0.490 (49 pages) and 0.330 (33 pages), respectively. The holdout base rate is 0.391, so the @50 lift is 1.18x for the model and 0.77x for the rule.

As model-only diagnostics, Logistic Regression has ROC AUC 0.700 and average precision 0.529. The random-ranking AP reference is the holdout prevalence, 0.391. These metrics were not calculated for the rule, so they do not establish that the model is superior in overall ranking.

The label is based on a change in impressions from the preceding 30 days to the latest 30 days. Pages with zero preceding-30-day impressions cannot be labeled `down`; results on the full holdout may therefore partly reflect label construction. On pages with a nonzero preceding baseline, model AUC is 0.611 and AP is 0.702 against a subset base rate of 0.606. The top-K overlap is also reported below because precision alone does not show how different the selected pages are.

For context, Week-4’s 0.420 Precision@50 was measured on all 30,000 pages; it is not the same evaluation as the rule’s 0.300 on this client holdout. These are within-window results from one split, not future predictions or proof of general superiority.

In [85]:
def week4_rule_score(frame):
    eligible = frame["content_age_days"].ge(180) & frame["impressions_90d"].ge(500)
    return frame["impressions_90d"].where(eligible, 0.0)


def rank_queue(frame, score_column, tie_breakers):
    sort_columns = [score_column] + [column for column, _ in tie_breakers]
    ascending = [False] + [is_ascending for _, is_ascending in tie_breakers]
    return frame.sort_values(
        sort_columns,
        ascending=ascending,
        kind="mergesort",
    ).reset_index(drop=True)


def precision_at_k(ranked_frame, k, label_column="is_declining_label"):
    selected = ranked_frame.head(min(k, len(ranked_frame)))
    return float(selected[label_column].mean()) if len(selected) else 0.0


def make_logistic_model(numeric_columns, categorical_columns, random_seed):
    other_numeric_columns = [column for column in numeric_columns if column != "avg_position"]
    other_numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    position_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", other_numeric_pipeline, other_numeric_columns),
            ("position", position_pipeline, ["avg_position"]),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
        ]
    )
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=random_seed,
                ),
            ),
        ]
    )


model = make_logistic_model(numeric_features, categorical_features, RANDOM_SEED)
model.fit(X_train, y_train)
model_score = model.predict_proba(X_test)[:, 1]
position_imputer = (
    model.named_steps["preprocessor"]
    .named_transformers_["position"]
    .named_steps["imputer"]
)
print(
    f"avg_position zeros treated as missing: {avg_position_missing_count:,}; "
    f"training median used for imputation: {position_imputer.statistics_[0]:.1f}; "
    "missing indicator enabled"
)

evaluation = df.iloc[test_index][
    [
        "content_id",
        "client_id",
        "content_age_days",
        "impressions_90d",
        "impressions_prev_30d",
        "is_declining_label",
    ]
].copy()
evaluation["model_score"] = model_score
evaluation["rule_score"] = week4_rule_score(evaluation)

model_ranked = rank_queue(evaluation, "model_score", [("content_id", True)])
rule_tie_breakers = [
    ("impressions_90d", False),
    ("content_age_days", False),
    ("content_id", True),
]
rule_ranked = rank_queue(evaluation, "rule_score", rule_tie_breakers)

k_values = [20, 50, 100]
base_rate_test = float(y_test.mean())
comparison_rows = [
    {
        "method": "Holdout prevalence reference",
        "precision@20": base_rate_test,
        "precision@50": base_rate_test,
        "precision@100": base_rate_test,
    }
]
for method_name, ranked in [
    ("Week-4 rule", rule_ranked),
    ("Logistic Regression", model_ranked),
]:
    row = {"method": method_name}
    for k in k_values:
        row[f"precision@{k}"] = precision_at_k(ranked, k)
    comparison_rows.append(row)

comparison = pd.DataFrame(
    comparison_rows,
    columns=["method", "precision@20", "precision@50", "precision@100"],
)
print("Primary comparison: Precision@K on the same client holdout")
print(comparison.round(3).to_string(index=False))
model_roc_auc = float(roc_auc_score(y_test, model_score))
model_average_precision = float(average_precision_score(y_test, model_score))
print(
    f"Model-only diagnostics (not computed for the rule): "
    f"ROC AUC={model_roc_auc:.3f}, average precision={model_average_precision:.3f}."
)
print(f"Full-data label base rate: {df['is_declining_label'].mean():.3f}")
print(f"Holdout label base rate: {base_rate_test:.3f} ({len(test_index):,} pages)")

for k in k_values:
    model_top_ids = set(model_ranked.head(k)["content_id"])
    rule_top_ids = set(rule_ranked.head(k)["content_id"])
    overlap = len(model_top_ids & rule_top_ids)
    print(
        f"Top-{k} overlap: {overlap}/{k} pages "
        f"({overlap / k:.1%}); model positives="
        f"{int(model_ranked.head(k)['is_declining_label'].sum())}/{k}; "
        f"rule positives={int(rule_ranked.head(k)['is_declining_label'].sum())}/{k}"
    )

    for method_name, ranked in [("model", model_ranked), ("rule", rule_ranked)]:
        queue = ranked.head(k)
        client_counts = queue["client_id"].value_counts()
        dominant_client = client_counts.index[0]
        holdout_client_share = evaluation["client_id"].eq(dominant_client).mean()
        print(
            f"  {method_name} top-{k}: {len(client_counts)} represented clients; "
            f"largest client share={client_counts.iloc[0] / k:.1%} "
            f"(that client's holdout share={holdout_client_share:.1%})"
        )

full_rule = df[
    ["content_id", "content_age_days", "impressions_90d", "is_declining_label"]
].copy()
full_rule["rule_score"] = week4_rule_score(full_rule)
full_rule_ranked = rank_queue(full_rule, "rule_score", rule_tie_breakers)
full_data_rule_p50 = precision_at_k(full_rule_ranked, 50)
print(f"Week-4 rule Precision@50 on all pages (not the client holdout): {full_data_rule_p50:.3f}")

rule_p50 = precision_at_k(rule_ranked, 50)
model_p50 = precision_at_k(model_ranked, 50)
print(f"Rule Precision@50 lift vs holdout base rate: {rule_p50 / base_rate_test:.3f}x")
print(f"Model Precision@50 lift vs holdout base rate: {model_p50 / base_rate_test:.3f}x")

# Sensitivity check: pages with a nonzero preceding-30-day impression baseline.
valid_baseline = evaluation["impressions_prev_30d"].gt(0)
valid_y = evaluation.loc[valid_baseline, "is_declining_label"].astype(int)
valid_scores = evaluation.loc[valid_baseline, "model_score"]
zero_baseline = ~valid_baseline
zero_baseline_declines = int(evaluation.loc[zero_baseline, "is_declining_label"].sum())
print(
    f"\nValid preceding-30-day baseline: {valid_baseline.sum():,}/"
    f"{len(evaluation):,} holdout pages ({valid_baseline.mean():.1%}); "
    f"declining labels among zero-baseline pages={zero_baseline_declines}"
)
if valid_y.nunique() == 2:
    print(f"AUC on valid-baseline holdout subset: {roc_auc_score(valid_y, valid_scores):.3f}")
    print(
        "AP on valid-baseline holdout subset: "
        f"{average_precision_score(valid_y, valid_scores):.3f}; "
        f"subset base rate={valid_y.mean():.3f}"
    )
else:
    print("Valid-baseline subset has only one label class; AUC/AP are not defined.")

avg_position zeros treated as missing: 1,205; training median used for imputation: 11.8; missing indicator enabled
Primary comparison: Precision@K on the same client holdout
                      method  precision@20  precision@50  precision@100
Holdout prevalence reference         0.391         0.391          0.391
                 Week-4 rule         0.350         0.300          0.330
         Logistic Regression         0.400         0.460          0.490
Model-only diagnostics (not computed for the rule): ROC AUC=0.700, average precision=0.529.
Full-data label base rate: 0.542
Holdout label base rate: 0.391 (2,325 pages)
Top-20 overlap: 0/20 pages (0.0%); model positives=8/20; rule positives=7/20
  model top-20: 2 represented clients; largest client share=95.0% (that client's holdout share=44.3%)
  rule top-20: 4 represented clients; largest client share=45.0% (that client's holdout share=47.6%)
Top-50 overlap: 1/50 pages (2.0%); model positives=23/50; rule positives=15/50
  model t

## 4. Errors and interpretation

On this single client holdout, Logistic Regression has higher Precision@20, Precision@50, and Precision@100 than the Week-4 rule. This is a result for this split, not evidence that the model is generally better: the top-50 queues overlap on only 1 page, and one client contributes 96% of the model's top-50 despite representing 44.3% of the holdout.

The largest absolute fitted coefficients are useful to inspect, but they are not independent feature-importance estimates or causal effects. Several inputs overlap or are correlated, and one-hot category coefficients are on a different scale from standardized numeric features.

The displayed hard cases show disagreements with the proxy label: the model gives high scores to some non-declining pages in its top-50 and very low scores to some declining pages ranked below it. All six displayed pages are younger than 180 days, so the Week-4 rule assigns each of these pages a score of zero because it falls below the age eligibility cutoff. These examples illustrate how the rule's eligibility condition and the model's learned ranking can treat the same pages differently. The three displayed non-declining pages have impressions above the full-data median, but are not in the extreme upper tail. All six examples have `engagement_rate = 0` and come from one held-out client; these are observed case details, not enough by themselves to establish why the model is wrong or that the client behaves differently. The label remains a current-window proxy, not a future outcome.

In [86]:
# Coefficients are inspectable, but are not independent feature importance.
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coef_values = model.named_steps["model"].coef_[0]
coef_frame = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": coef_values,
        "abs_coefficient": np.abs(coef_values),
    }
).sort_values("abs_coefficient", ascending=False)

print("Largest absolute fitted coefficients (not a feature-importance ranking):")
print(coef_frame.head(10)[["feature", "coefficient"]].round(3).to_string(index=False))

# Ranking errors: non-declining pages in the model's top-50 queue and declining pages below it.
model_top_50 = model_ranked.head(50).copy()
queue_false_positives = (
    model_top_50[model_top_50["is_declining_label"].eq(0)]
    .sort_values(["model_score", "content_id"], ascending=[False, True], kind="mergesort")
    .head(3)
)
missed_declines = (
    model_ranked.iloc[50:][model_ranked.iloc[50:]["is_declining_label"].eq(1)]
    .sort_values(["model_score", "content_id"], ascending=[True, True], kind="mergesort")
    .head(3)
)

hard_cases = pd.concat(
    [
        queue_false_positives.assign(case_type="non-declining page in model top-50"),
        missed_declines.assign(case_type="declining page ranked below model top-50"),
    ],
    ignore_index=True,
)
hard_cases["engagement_rate"] = hard_cases["content_id"].map(
    df.set_index("content_id")["engagement_rate"]
)
hard_cases["sessions_90d"] = hard_cases["content_id"].map(
    df.set_index("content_id")["sessions_90d"]
)
hard_cases["impressions_percentile"] = hard_cases["content_id"].map(
    df.set_index("content_id")["impressions_90d"].rank(pct=True)
)

print("\nThree non-declining pages in the model's top-50 queue:")
print(
    queue_false_positives[
        ["content_id", "model_score", "content_age_days", "impressions_90d", "rule_score"]
    ].to_string(index=False)
)
print("\nThree declining pages ranked below the model's top-50 queue:")
print(
    missed_declines[
        ["content_id", "model_score", "content_age_days", "impressions_90d", "rule_score"]
    ].to_string(index=False)
)

print(
    f"\nHard-case facts: {int(hard_cases['content_age_days'].lt(180).sum())}/"
    f"{len(hard_cases)} are below age 180; "
    f"{int(hard_cases['rule_score'].eq(0).sum())}/{len(hard_cases)} receive rule score zero; "
    f"{int(hard_cases['engagement_rate'].eq(0).sum())}/{len(hard_cases)} have engagement_rate=0."
)
print(f"Full-data median impressions_90d: {df['impressions_90d'].median():.1f}")
print(f"Full-data maximum impressions_90d: {df['impressions_90d'].max():,.0f}")
print(
    "Impression percentiles for these six examples: "
    f"{hard_cases['impressions_percentile'].round(3).tolist()}"
)

if not hard_cases.empty:
    case_client_counts = hard_cases["client_id"].value_counts()
    dominant_case_client = case_client_counts.index[0]
    dominant_case_rows = int(evaluation["client_id"].eq(dominant_case_client).sum())
    print(
        f"Examples from the most represented case client: {case_client_counts.iloc[0]}/"
        f"{len(hard_cases)}; its holdout size={dominant_case_rows:,}/"
        f"{len(evaluation):,} ({dominant_case_rows / len(evaluation):.1%})."
    )

# Candidate explanations to inspect, not conclusions: volumes, client mix, and engagement missingness.
print("\nHard-case values for follow-up review:")
print(
    hard_cases[
        [
            "case_type",
            "content_id",
            "model_score",
            "content_age_days",
            "impressions_90d",
            "impressions_percentile",
            "sessions_90d",
            "engagement_rate",
            "is_declining_label",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

Largest absolute fitted coefficients (not a feature-importance ranking):
                                feature  coefficient
                        num__word_count        1.669
               num__log_impressions_90d        1.416
                        num__char_count       -1.415
position__missingindicator_avg_position       -0.731
                    num__log_clicks_90d       -0.629
         cat__impression_tier_excellent       -0.555
               cat__impression_tier_low        0.545
         cat__word_count_tier_2000-3500       -0.455
               cat__main_intent_unknown        0.445
                 position__avg_position       -0.422

Three non-declining pages in the model's top-50 queue:
          content_id  model_score  content_age_days  impressions_90d  rule_score
content_8fdbff16a886     0.886526               175             3863           0
content_e354f8e518c2     0.861462               175             1163           0
content_da806b9f243e     0.857258            

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.